# Instrument an Existing App and Verify It End to End

Add tracing to an app that has none, then prove it worked with ten machine-checked gates that either pass or name exactly what is missing.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/future-agi/cookbooks/blob/cookbook/quickstart-notebooks/quickstart/instrument-and-verify.ipynb)

The app here is `examples/customer_service` from [openai/openai-agents-python](https://github.com/openai/openai-agents-python): an airline support agent with a triage agent, two specialist agents, two tools and a handoff. Nothing in it was written for this notebook.


## Install

`traceai-openai-agents` is the instrumentor for this app's SDK. Swap it for `traceai-openai`, `traceai-langchain`, `traceai-llamaindex` and the rest when the app uses something else.


In [ ]:
%pip install fi-instrumentation-otel traceai-openai-agents openai-agents --quiet

In [ ]:
import os

os.environ["FI_API_KEY"] = "your-api-key"            # app.futureagi.com -> Keys
os.environ["FI_SECRET_KEY"] = "your-secret-key"
os.environ["FI_PROJECT_NAME"] = "support-agent-quickstart"
os.environ["OPENAI_API_KEY"] = "your-openai-api-key"  # the app needs it; tracing does not

os.environ["FI_VERIFY"] = "1"
os.environ["FI_MODEL_RATES"] = '{"gpt-4o-mini": [0.15, 0.60]}'   # {"<model>": [in, out]} per 1M
os.environ["AGENT_MODEL"] = "gpt-4o-mini"

## Get the app and the checker

`fi_verify.py` is one file with no dependency beyond the OpenTelemetry SDK. It is the only thing that decides the result.


In [ ]:
!git clone --depth 1 https://github.com/openai/openai-agents-python 2>/dev/null
%cd openai-agents-python
!mkdir -p observability/futureagi && touch observability/__init__.py observability/futureagi/__init__.py
!curl -fsSL https://docs.futureagi.com/fi_verify.py -o observability/futureagi/fi_verify.py
!shasum -a 256 observability/futureagi/fi_verify.py
# 9b331742bc4143a46ae22ee0e8a1c8487d70cc242f689491cc20ca94a336ecd0

## Step 1: Prove the keys before writing code that depends on them

`preflight` sends one real span, then three deliberately broken variants. If any broken variant is accepted, nothing is proven and it fails.


In [ ]:
!python observability/futureagi/fi_verify.py preflight

## Step 2: Make one request arrive as one tree

`register()` puts `project_name` and `project_type` on the resource, which is G2. `FITracer`, not `get_tracer()`: a plain `Tracer` drops `session.id` and `user.id`, failing G6 and G7.


In [ ]:
%%writefile observability/futureagi/setup.py
import os, sys
from opentelemetry import trace
from fi_instrumentation import FITracer, register
from fi_instrumentation.fi_types import ProjectType
from traceai_openai_agents import OpenAIAgentsInstrumentor

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))   # for fi_verify

# Import once at process start, AFTER whatever loads your .env and before any model call.
tracer_provider = None
tracer = trace.get_tracer(__name__)   # a no-op tracer, so no key means no spans, not a crash

if os.getenv("FI_API_KEY") and os.getenv("FI_SECRET_KEY"):
    tracer_provider = register(
        project_name=os.getenv("FI_PROJECT_NAME", "my-app"),
        project_type=ProjectType.OBSERVE,
        set_global_tracer_provider=True,
    )
    OpenAIAgentsInstrumentor().instrument(tracer_provider=tracer_provider)
    # FITracer, not get_tracer(): a plain Tracer drops session.id and user.id, failing G6 and G7
    tracer = FITracer(tracer_provider.get_tracer(__name__))
    if os.getenv("FI_VERIFY") == "1":
        import fi_verify; fi_verify.attach(tracer_provider)

## Step 4: Write the attributes behind each field, cost included

Cost is never calculated for you. It is billed per LLM span as each one ends and summed onto the root, which is what the trace list reads.

Written here, before Step 3, because the entry point imports the roll-up module.


In [ ]:
%%writefile observability/futureagi/futureagi_rollup.py
import json, os, contextvars
from opentelemetry import context as otel_context
from opentelemetry.sdk.trace import SpanProcessor
from observability.futureagi.setup import tracer_provider as P   # None until the keys are set

RATES = json.loads(os.getenv("FI_MODEL_RATES", "{}"))   # {"<model>": [in, out]} per 1M
MODEL = contextvars.ContextVar("fi_model", default=None)   # what llm_call is about to call
RUN = otel_context.create_key("fi_run")     # rides the OpenTelemetry context, so a copied
M = "gen_ai.request.model"                  # context carries it over a thread hand-off too.
T = ["gen_ai.usage.input_tokens", "gen_ai.usage.output_tokens",
     "gen_ai.usage.total_tokens", "gen_ai.cost.total"]


class RollUp(SpanProcessor):
    def on_start(self, span, parent_context=None):   # the instrumentor never leaves its LLM
        if MODEL.get(): span.set_attribute(M, MODEL.get())   # span current in your code, so
    def on_end(self, span):                          # the name goes on as the span opens
        run, a = otel_context.get_value(RUN), span.attributes or {}
        if run is not None and T[0] in a:      # tokens land on LLM spans, which end first
            i, o = RATES.get(a.get(M), [0, 0])            # the only place a cost is
            a = dict(a, **{T[3]: (a[T[0]] * i + a.get(T[1], 0) * o) / 1e6})   # ever computed
        for k in T: run[k] = run.get(k, 0) + a.get(k, 0)


# register() leaves _default_processor set, and the first add_span_processor call on that
# provider discards the exporter it installed. Clearing it first is what keeps delivery.
if P: P._default_processor = False; P.add_span_processor(RollUp())

In [ ]:
%%writefile observability/futureagi/futureagi_spans.py
from contextlib import contextmanager
from observability.futureagi.futureagi_rollup import MODEL


@contextmanager               # wrap the client call. The instrumentor writes the rest
def llm_call(model):
    t = MODEL.set(model)
    try: yield
    finally: MODEL.reset(t)

## Step 3: Carry session and user from one scope at the edge

`main.py` is an interactive REPL, so this repository has no entry point that runs once. This is that entry point: the roll-up and the scope outside, the `CHAIN` root inside, a flush before returning.


In [ ]:
%%writefile examples/customer_service/run_traced.py
"""One customer message through the airline support agent, traced end to end.

main.py is an interactive REPL, so the repository has no entry point that runs once. This
is that entry point: the roll-up and the scope outside, the CHAIN root inside, a flush
before returning. Run it twice with the same FI_SESSION_ID to see a session of two traces.
"""
from __future__ import annotations

import asyncio
import os
import sys
import uuid

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__)))))

from observability.futureagi.setup import tracer, tracer_provider   # noqa: E402
from observability.futureagi import futureagi_rollup as R           # noqa: E402
from fi_instrumentation import using_attributes                     # noqa: E402
from opentelemetry import context as otel_context                   # noqa: E402

from agents import Runner, RunConfig, set_default_openai_api        # noqa: E402
from examples.customer_service.main import AirlineAgentContext, triage_agent   # noqa: E402

if os.getenv("OPENAI_BASE_URL"):      # an OpenAI-compatible endpoint that is not OpenAI
    set_default_openai_api("chat_completions")   # only OpenAI serves the Responses API

MODEL = os.getenv("AGENT_MODEL", "gpt-4o-mini")
DEFAULT = "How much baggage am I allowed to bring on the plane?"


async def handle(question, session_id, user_id):
    """One customer message: the roll-up and the scope outside, the root inside."""
    run = {k: 0 for k in R.T}          # mutated in place, so a copied context shares it
    token = otel_context.attach(otel_context.set_value(R.RUN, run))
    try:
        with using_attributes(session_id=session_id, user_id=user_id,
                              tags=["prod"], metadata={"channel": "web"}):
            with tracer.start_as_current_span("support.turn") as root:
                root.set_attribute("gen_ai.span.kind", "CHAIN")
                root.set_attribute("input.value", question[:8000])
                root.set_attribute(R.M, MODEL)   # the Model column reads the root, not the
                                                 # LLM spans the instrumentor writes it on
                result = await Runner.run(
                    triage_agent, [{"content": question, "role": "user"}],
                    context=AirlineAgentContext(), run_config=RunConfig(model=MODEL))
                answer = str(result.final_output)
                root.set_attribute("output.value", answer[:8000])
                run[R.T[2]] = run.get(R.T[2]) or run.get(R.T[0], 0) + run.get(R.T[1], 0)
                for k in R.T: root.set_attribute(k, run.get(k, 0))
                return result, answer
    finally:
        otel_context.detach(token)


async def main():
    question = " ".join(sys.argv[1:]) or DEFAULT
    session_id = os.getenv("FI_SESSION_ID") or "conv_" + uuid.uuid4().hex[:12]
    user_id = os.getenv("ACCOUNT_ID", "acct_10427")

    print(f"customer: {question}")
    result, answer = await handle(question, session_id, user_id)
    print(f"{result.last_agent.name}: {answer}")

    if tracer_provider: tracer_provider.force_flush()   # a batch sends on a timer
    print(f"\nsession {session_id}  user {user_id}")


if __name__ == "__main__":
    asyncio.run(main())

## Step 5: Run the ten gates

One real request through your entry point, then `check`. It exits `0`, or it names the gate that failed and why.


In [ ]:
import os, uuid
os.environ["FI_SESSION_ID"] = "conv_" + uuid.uuid4().hex[:12]
print(os.environ["FI_SESSION_ID"])

In [ ]:
!python examples/customer_service/run_traced.py "How much baggage am I allowed to bring on the plane?"

In [ ]:
!python observability/futureagi/fi_verify.py check

Run it again with the same `FI_SESSION_ID` and a second message, and the two turns join one session in the dashboard.


In [ ]:
!python examples/customer_service/run_traced.py "Thanks. Please move me to seat 12A, my confirmation number is LL0EHY."

## Step 6: Bind evals to attributes the trace now carries

An eval can only grade what it is bound to, and you bind every variable yourself. Read your own attribute paths out of the capture first, then attach the evals in the console.

| Eval | Scope | Bound to |
|---|---|---|
| Task Completion | Traces | `input.value` and `output.value` on the root |
| Evaluate Function Calling | Spans | the tool call arguments on the first LLM span |
| Detect Hallucination | Traces | `input.value` and `output.value` |
| Instruction Adherence | Traces | `input.value` and `output.value` |
| PII Detection | Spans | `output.value` |


In [ ]:
import json

paths = set()
for line in open(".fi_verify/spans.jsonl"):
    paths.update(json.loads(line)["attrs"])
for p in sorted(paths):
    print(p)

## What you built

An integration whose correctness is decided by a machine rather than by looking at a dashboard. `fi_verify.py check` exits 0 only when all ten gates pass, never nine.

Next: [Setup evals](https://docs.futureagi.com/docs/observe/guides/setup-evals) to attach the five above, or [Distributed Tracing](https://docs.futureagi.com/docs/cookbook/quickstart/distributed-tracing) to keep one trace across a service boundary.
